In [1]:
 import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(96, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
train_full = torchvision.datasets.STL10(
    root="./data",
    split="train",
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.STL10(
    root="./data",
    split="test",
    download=True,
    transform=transform_test
)

 36%|███▋      | 958M/2.64G [23:12<39:17, 714kB/s]    

In [ ]:
val_size = int(0.2 * len(train_full))
train_size = len(train_full) - val_size

train_dataset, val_dataset = random_split(
    train_full,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

plt.imshow(np.transpose(images[0].numpy(), (1, 2, 0)))
plt.title(f"Label: {labels[0]}")
plt.show()

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 24 * 24, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
def train_model(model, train_loader, val_loader, epochs=5):
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                loss = criterion(outputs, y)
                val_loss += loss.item()

        history["train_loss"].append(train_loss / len(train_loader))
        history["val_loss"].append(val_loss / len(val_loader))

        print(f"Epoch {epoch+1}: train={history['train_loss'][-1]:.4f}, val={history['val_loss'][-1]:.4f}")

    return history

In [ ]:
cnn_model = SimpleCNN()
history_cnn = train_model(cnn_model, train_loader, val_loader)

In [ ]:
plt.plot(history_cnn["train_loss"], label="train")
plt.plot(history_cnn["val_loss"], label="val")
plt.legend()
plt.title("CNN Loss")
plt.show()

In [ ]:
from torchvision.models import resnet18

model_resnet = resnet18(pretrained=True)

for param in model_resnet.parameters():
    param.requires_grad = False

model_resnet.fc = nn.Linear(model_resnet.fc.in_features, 10)
model_resnet = model_resnet.to(device)

In [ ]:
history_resnet = train_model(model_resnet, train_loader, val_loader)

In [ ]:
plt.plot(history_cnn["val_loss"], label="CNN")
plt.plot(history_resnet["val_loss"], label="ResNet")
plt.legend()
plt.title("Comparison")
plt.show()

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn

model_det = fasterrcnn_resnet50_fpn(pretrained=True)
model_det.eval()

In [ ]:
import requests
from PIL import Image

url = "https://ultralytics.com/images/zidane.jpg"
image = Image.open(requests.get(url, stream=True).raw)

transform = transforms.ToTensor()
img_tensor = transform(image)

with torch.no_grad():
    prediction = model_det([img_tensor])

print(prediction[0].keys())

In [ ]:
boxes = prediction[0]['boxes']
scores = prediction[0]['scores']

plt.imshow(image)

for i in range(len(boxes)):
    if scores[i] > 0.7:
        box = boxes[i].numpy()
        plt.gca().add_patch(
            plt.Rectangle(
                (box[0], box[1]),
                box[2]-box[0],
                box[3]-box[1],
                fill=False,
                edgecolor='red',
                linewidth=2
            )
        )

plt.show()

In [ ]:
# HW10-11 Report

## 1. Dataset

Использован STL10 через torchvision.datasets.
Разделение: 80% train / 20% val.

## 2. CNN

Реализована простая CNN:
Conv -> ReLU -> Pool -> Conv -> ReLU -> Pool -> FC

Результаты:
- модель обучается стабильно
- наблюдается уменьшение loss

## 3. Transfer Learning (ResNet18)

Использована pretrained ResNet18:
- заморожен backbone
- обучается только head

Результат:
- быстрее сходится
- даёт лучший val loss

## 4. Detection (S11)

Использована Faster R-CNN pretrained модель.

Результат:
- корректно обнаруживает объекты
- bounding boxes отображены

## 5. Вывод

- CNN работает, но уступает pretrained моделям
- transfer learning даёт значительное улучшение
- detection требует других метрик и подходов